# Continue the completed 06b baseline with budgeted A/B/C/D training

This continuation reuses the completed **sub-01, seed-0 route_c baseline** and its prepared training-only captions. Existing baseline, captions, preflight, and profile artifacts are retained. Training stays disabled by default. When intentionally enabled, selected A/B/C/D arms run under a soft time budget in a new output directory, using the original 60-epoch schedule and resumable checkpoints. No official-test evaluation is included.


In [ ]:
# Bootstrap the audited sources before importing torch or any project module.
import hashlib, json, os, pathlib, subprocess, sys
if any(name == 'phase1' or name.startswith('phase1.') for name in sys.modules):
    raise RuntimeError('phase1 is already imported. Restart the runtime before this bootstrap so audited source hashes also identify the code being executed.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
try:
    from google.colab import drive
except ImportError:
    pass  # Local CPU execution uses the same cells with the environment paths below.
else:
    drive.mount('/content/drive')
BRANCH='research/trimodal-budgeted-arms-20261121'
REPO=pathlib.Path(os.environ.get('EEG_06C_REPO', '/content/eeg-alignment-phase1-06c-budgeted'))
# The branch is a delivery location; these LF-normalized hashes pin the Python sources.
# Regenerate only after reviewing/testing an intentional source change.
EXPECTED_SOURCE_SHA256 = {'phase1/__init__.py': '7c6a15363d636c811994e4a796a59f900f26d697022647aee7a3bb3b6a7fb441',
 'phase1/core.py': 'd7e3c9829cf55d1fcad3465f64ae36ea3342d6812c944ad0a34c613bc77c5cdd',
 'phase1/experiment.py': 'fd2543be8795f934a73ce1a604d8b0f3ca5fa127c43f1108cb2aead3eb448973',
 'phase1/extraction_environment.py': 'ba0fa4f3f4e7c93999d05588d884d44758c696deb099c455a1544c0ad6f580fc',
 'phase1/final_audit.py': '34ec2dc1174b62c161e85bbfac75932a61d2c6de6d22f061774140e07a6f1d93',
 'phase1/historical_06b.py': 'd903470a0b790cd416494fd5e57a5fea6a6243d94e49820610603b2e44de3a28',
 'phase1/input_diagnostics.py': '8169b40154f337cb354c55eb6caa38d4e791599e087d18056575c5a7182ae1cf',
 'phase1/joint_data.py': 'e4acced0c8e238891335261b8551ffe928d0cd7cc026eb8586a711343e1d4630',
 'phase1/joint_experiment.py': '7e7032837ce781ad26c196019f0f18e9d2bb2b8526cb85a704c62452f95e30ce',
 'phase1/joint_inference.py': 'a2a95125d6b5e4fd8abf4b6b427f246515af53153b9386bb6ebe14b2f38e9651',
 'phase1/joint_model.py': 'cf4bbe3c8f0a2c67dbc59bb7ef2cd94fe171cd7a3a2caa875d3889b51f92657c',
 'phase1/joint_smoke.py': 'ca23d172fe66ceb83df5b9b0515448522134c5952fb44725fa0e2fe71ff2f26f',
 'phase1/models.py': 'c080717e68643f57cc78ab952b74d7d026a0857f5ff663133858575a38f56ab9',
 'phase1/patch_adapter.py': '9d5935add9b695982a8a04f9450c84c5ddc2215e8cf81386293538f0506e895a',
 'phase1/patch_cache.py': '163f0d7b565dbdb0f63a671f3c60cc015e2162c67066b26fa06adb8e5c5e5d29',
 'phase1/patch_experiment.py': '0b4e36a687456610c386e43b8114c0e6ca0d2b668890d69373649c9311f94c19',
 'phase1/patch_prepare.py': '5584969828298eb9c1b6c2fcea5bb43f2a7a34afdf416ac59bee3b6147c40e46',
 'phase1/patch_resources.py': '4cd7fda55a14efa4a8c298528a4936d4cfbd0e3265f4982da65fb932fe6ca058',
 'phase1/pilot_inputs.py': 'ea5e1ce03cb3022be701cef1adb97c4601d0c126b25ff685a59d9d7ab96459cb',
 'phase1/pilot_preflight.py': '90155a155c330d61902faed35defeb5343394f7f0e346576c4f8718f54ae2044',
 'phase1/process_log.py': '33da9bfcc7f4c565f792e206894108ffa260bd80e6a275de93ab06d56449a94f',
 'phase1/reconcile.py': '7e561ec8d0bb2fa80f8f986118bdd6a35cfb31aaa3d34e780c8f44a6a09dfa9b',
 'phase1/semantic.py': 'ef6a71155985d499ff9a3f68a1619eb072958477374a50a5f21598f406104aaa',
 'phase1/smoke.py': '75b55d8c8b4ce0de58c3c1ff163bbfa2210384253b401ff0fa5a87393cc897fc',
 'phase1/training_preflight.py': 'b8ab5fffc16aab413338864279942fa04036ecb41b9a39456a93cf75f7e998fe',
 'phase1/trimodal_06b.py': '6fa779798bb8620ff939ca2cb1d2ce51f710ae81ae4119ec7f8bc90af53b07b6',
 'phase1/v7_controls.py': '3ce8d6a8f49db9f3f813c8e3a125fb3eb568e00af5bc7b8628c5de111e9734c8',
 'phase1/validation_root_audit.py': '6d9aa0d693c88132c2a57d1dd5b00f71de2eed6c44114c318009b8769a91ae31'}
def verify_audited_sources(repo):
    actual={path.relative_to(repo).as_posix():hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
            for path in sorted((repo/'phase1').rglob('*.py'))}
    if actual != EXPECTED_SOURCE_SHA256:
        raise RuntimeError('Audited source hash mismatch. Use the notebook and code from the same verified audit commit.')

if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
else:
    branch=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    if branch!=BRANCH or dirty: raise RuntimeError('Existing notebook checkout is on another branch or has local changes; preserve it and use a fresh runtime/path.')
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
if actual != BRANCH: raise RuntimeError(f'Expected {BRANCH}, got {actual}')
verify_audited_sources(REPO)
sys.path.insert(0,str(REPO))
import numpy as np, torch
print('branch:',actual,'commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
print('PyTorch:',torch.__version__,'NumPy:',np.__version__,'CUDA available:',torch.cuda.is_available())


## Required full historical training cache

Default cache root: `/content/drive/MyDrive/eeg_results/cache`. Required: `sub-01_train_z16.npy` with shape `[16540,63,time]`, plus `feat6_internvit6b_parts/train_000.pt` through `train_008.pt`. Each train chunk must contain mean-pooled layers 20/24/28/32/36, width 3200, aligned in the original 06b training image order. The CPU preflight rejects missing, changed, nonfinite, wrong-shape, or wrong-protocol inputs. Hashes and shapes do not establish EEG channel semantics or image-feature row identity; those remain dependent on independent cache provenance. It never falls back to the 2,000-image pilot or opens test files. See `docs/HISTORICAL_06B_CHECKPOINTED_REPRODUCTION.md` for exact requirements and the optional train-only caption artifact contract.

In [ ]:
from phase1.historical_06b import preflight, profile_cpu
CACHE=pathlib.Path(os.environ.get('EEG_CACHE', '/content/drive/MyDrive/eeg_results/cache'))
RUN_ROOT=pathlib.Path(os.environ.get('EEG_06C_RUN_ROOT', '/content/drive/MyDrive/eeg_results/v6b_checkpointed_phase1/sub01_seed0_routec'))
MANIFEST=RUN_ROOT/'train_only_manifest.json'
BASELINE_OUTPUT=RUN_ROOT/'baseline'
CPU_CHECK_OUTPUT=RUN_ROOT/'budgeted_cpu_checks_20261121'
RUN_CPU_PREFLIGHT=False  # Existing verified inputs are rechecked by the runner when enabled.
if RUN_CPU_PREFLIGHT:
    if not MANIFEST.is_file(): raise FileNotFoundError('Reuse the completed baseline manifest; this continuation does not recreate it.')
    preflight_report=preflight(MANIFEST)
    CPU_CHECK_OUTPUT.mkdir(parents=True,exist_ok=True)
    (CPU_CHECK_OUTPUT/'cpu_preflight.json').write_text(json.dumps(preflight_report,indent=2))
    print(json.dumps(preflight_report,indent=2))
else:
    print('Existing baseline CPU preflight retained; standalone preflight skipped.')


In [ ]:
# Optional CPU check; existing profile artifacts are never overwritten.
RUN_CPU_PROFILE=False
if RUN_CPU_PROFILE:
    profile=profile_cpu(MANIFEST,batch_size=32)
    CPU_CHECK_OUTPUT.mkdir(parents=True,exist_ok=True)
    (CPU_CHECK_OUTPUT/'cpu_profile.json').write_text(json.dumps(profile,indent=2))
    print(json.dumps(profile,indent=2))
    assert profile['official_test_opened'] is False
else:
    print('Existing CPU profile retained; repeated CPU updates skipped.')


## Reuse the completed historical 06b baseline

The completed baseline uses historical posterior-17 EEG, mean-pooled InternViT layers 20/24/28/32/36, the original Aligner and normalization, and AdamW/OneCycleLR with 60 epochs and batch 1,024. Its original manifest, `baseline/run.json`, and best checkpoint are inputs to the tri-modal runner and remain unchanged. This continuation does not reproduce or resume baseline training.


In [ ]:
RUN_BASELINE=False  # This continuation preserves the completed baseline.
if RUN_BASELINE:
    raise RuntimeError('Baseline reruns are disabled in this continuation. Keep RUN_BASELINE=False and reuse the verified completed baseline.')
else:
    print('Completed baseline retained at:', BASELINE_OUTPUT)


## Budgeted matched A/B/C/D from the existing checkpoint

The runner verifies the completed baseline and the three existing training-only caption inputs before training. It never prepares or rewrites those inputs. A/B/C/D retain identical initialization, split, optimizer, 60-epoch scheduler, and validation cadence. Select any unfinished arms with `ARMS`; `TIME_BUDGET_SECONDS=900` is a soft per-invocation budget and checkpoint writes may finish after the limit.

The first intentional execution should use `PROFILE=True`, `ARMS=('A','B','C','D')`. The absolute five-epoch ceiling measures one complete validation interval per arm while consuming the beginning of the actual experiment. If the budget stops early, rerun with the same settings to finish that interval. The profile summary remains NO-GO for a full runtime estimate until all four arms have measured CUDA timings through epoch 5. Review that measured timing and memory report before continuing; partial or CPU-only profiles do not supply a GPU estimate. After a GO decision, set `PROFILE=False` and resume selected arms in **the same new output directory** through the fixed 60-epoch schedule. Do not use the timing profile to choose model hyperparameters, a winning arm, or a new seed. No checkpoint is restarted for profiling, and the old `trimodal_abcd` directory is never reused.


In [ ]:
from phase1.trimodal_06b import preflight_text_inputs, run as run_trimodal
TEXT_FEATURES=CACHE/'captions_train_only.npy'
CAPTION_IDS=CACHE/'caption_ids_train_only.json'
CAPTION_PROVENANCE=CACHE/'caption_provenance_train_only.json'
caption_files=[TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE]
RUN_CAPTION_PREFLIGHT=False  # Runner always verifies these inputs before an enabled run.
if RUN_CAPTION_PREFLIGHT:
    print(json.dumps(preflight_text_inputs(*caption_files),indent=2))
else:
    print('Existing captions retained; standalone caption preflight skipped.')


In [ ]:
RUN_TRIMODAL=False  # Intentionally enable only after reviewing the settings below.
PROFILE=True        # First five epochs of the actual experiment; keep its checkpoints.
ARMS=('A','B','C','D')  # Select unfinished arms in either mode.
TIME_BUDGET_SECONDS=900.0  # Soft total budget per invocation, not a GPU runtime prediction.
TRIMODAL_OUTPUT=RUN_ROOT/'trimodal_budgeted_20261121'
if RUN_TRIMODAL:
    import math
    if not isinstance(TIME_BUDGET_SECONDS, (int,float)) or not math.isfinite(TIME_BUDGET_SECONDS) or TIME_BUDGET_SECONDS <= 0:
        raise ValueError('Use a finite positive TIME_BUDGET_SECONDS.')
    if not isinstance(PROFILE, bool): raise ValueError('PROFILE must be True or False.')
    if not torch.cuda.is_available(): raise RuntimeError('An enabled budgeted run requires a Colab GPU.')
    if not all(path.is_file() for path in caption_files): raise FileNotFoundError('Reuse the prepared train-only caption artifacts; mixed caches are not accepted.')
    records=run_trimodal(MANIFEST,BASELINE_OUTPUT,TEXT_FEATURES,CAPTION_IDS,CAPTION_PROVENANCE,
                        TRIMODAL_OUTPUT,'cuda',resume=True,arms=ARMS,
                        max_seconds=TIME_BUDGET_SECONDS,max_epochs_per_arm=5 if PROFILE else None)
    print(json.dumps(records,indent=2))
    if PROFILE:
        from phase1.trimodal_06b import summarize_gpu_profile
        print(json.dumps(summarize_gpu_profile(TRIMODAL_OUTPUT),indent=2))
else:
    print('Budgeted A/B/C/D training is disabled. Existing inputs and results remain unchanged.')
    print('Next enabled destination:', TRIMODAL_OUTPUT)


## Scientific boundary

The historical test evaluation is intentionally absent. Baseline selection and tri-modal reporting use training-partition validation only. The preserved historical image-feature normalization is fitted on all 16,540 training-partition rows, including the 1,654 validation rows; this validation is not isolated from that preprocessing. The new text normalizer and residualizer fit only optimizer rows. Cache hashes and tensor shapes prove file stability and structural compatibility, not image-row identity or the original EEG channel order. The original 06b result and this development validation are not an untouched-test comparison. This notebook does not claim SOTA, significance, or a tri-modal gain. Preserve each `run.json`, input hash report, CPU profile, epoch log, and checkpoint with the Drive output.